In [ ]:
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import os
from pathlib import Path

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=0.000001):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, forward_output, target):
        probs = torch.sigmoid(forward_output)

        probs = probs.view(-1)
        target = target.view(-1)

        intersection = (probs * target).sum()
        union = (probs + target).sum()
        dice_loss = 1 - (2 * intersection + self.smooth) / (union + self.smooth)

        return dice_loss

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super(BCEDiceLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight

    def forward(self, forward_output, target):
        bce_loss = self.bce(forward_output, target)
        dice_loss = self.dice(forward_output, target)
        dice_weight = 1 - self.bce_weight
        bce_weight = self.bce_weight

        loss = bce_loss * bce_weight + dice_loss * dice_weight

        return loss

In [ ]:
class H5Dataset(Dataset):
    def __init__(self, h5_path, mode='train', augment=False, seed = 77): #CPBL UNI LIONS AK 77 #mode = '(.h5 group)'
        self.h5_path = h5_path
        self.mode = mode
        self.augment = augment
        self.seed =seed
        self.current_epoch = 0 #epoch counter

    def set_epoch(self, epoch):
        self.current_epoch = epoch

    def __getitem__(self, index): #for dataset[index]
        with h5py.File(self.h5_path, 'r') as f:
            img = f[f'{self.mode}/images'][index] #img.shape = (256, 256, 3) = (H, W, C)
            mask = f[f'{self.mode}/masks'][index] #mask.shape = (256, 256) = (H, W)

        if self.augment and self.mode == 'train':
            rng = random.Random(self.seed + index + self.current_epoch) #格式random.Random(seed)
            k = rng.randint(0, 3) #0到3隨機int
            img = np.ascontiguousarray(np.rot90(img, k, axes=(0, 1)))
            mask = np.ascontiguousarray(np.rot90(mask, k, axes=(0, 1)))
            #np.ascontiguousarray整理記憶體位置 #np.rot90(target, 次數, 參與旋轉的維度)旋轉np矩陣一次90度 #axes=(0, 1)指img, mask .shape的(H, W, C), (H, W)對應索引位置(0, 1, 2), (0, 1)
            brightness_factor = rng.uniform(0.8, 1.2)
            img = (img.astype(np.float32) * brightness_factor)
            img = np.clip(img, 0, 255).astype(np.uint8) #np.clip(target, min, max) 限制數據範圍超過max變max, 低於min變min

        img_t = torch.from_numpy(img[:, :, 0]).float().unsqueeze(0) / 255.0 #img.shape = (256, 256, 3) >>> img[:, :, 0] = (256, 256) >>> .unsqueeze(0) = (1, 256, 256) = (C, H, W)
        mask_t = torch.from_numpy(mask).float().unsqueeze(0) / 255.0

        return img_t, mask_t

    def __len__(self):
        with h5py.File(self.h5_path, 'r') as f:
            return len(f[f'{self.mode}/images'])

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        def conv_block(in_channels, out_channels, kernel_size, stride, padding):
            return nn.Sequential(
                nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size, stride=1, padding=padding),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )

        #encoder
        self.enc1 = conv_block(in_channels, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.enc2 = conv_block(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)

        #decoder
        self.up = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=2, stride=2) #image size = (128*128) >>> (256, 256)

        self.dec = conv_block(in_channels=128, out_channels=64, kernel_size=3, stride=1, padding=1) #combine feature channels = enc1(64)+up(64)

        self.final = nn.Conv2d(in_channels=64, out_channels=out_channels, kernel_size=1)

    def forward(self,x):
        enc1 = self.enc1(x)
        pool = self.pool(enc1)
        enc2 = self.enc2(pool)

        up = self.up(enc2)

        tiger = torch.cat([up, enc1], dim=1) #(Batch, Channel, Height, Width) so dim=1 代表合併的是C
        dec = self.dec(tiger)

        forward = self.final(dec)
        return forward

In [ ]:
def train_model(h5_path, batch, epochs, seed, learning_rate):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed) #固定weight initiation的隨機數字
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training device: {device}")

    train_dataset = H5Dataset(h5_path=h5_path, mode='train', augment=True, seed=seed)
    val_dataset = H5Dataset(h5_path=h5_path, mode='val', augment=False, seed=seed)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch, shuffle=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch, shuffle=False)

    model = UNet(in_channels=1, out_channels=1).to (device)
    loss_function = BCEDiceLoss(bce_weight=0.5)
    optimizer = optim.Adam(model.parameters(), learning_rate) #依權重過去更新狀況微調當次learning rate

    final_val_loss = 0.0
    final_val_acc = 0.0

    for epoch in range(epochs):
        train_loader.dataset.set_epoch(epoch) #train_loader.dataset指向H5Dataset #呼叫set_epoch(epoch)

        model.train()
        train_loss = 0.0
        for img, masks in train_loader:
            img, masks = img.to(device), masks.to(device)
            optimizer.zero_grad()
            output = model(img)
            loss = loss_function(output, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        correct_pixels = 0
        total_pixels = 0
        with torch.no_grad():
            for img, masks in val_loader:
                img, masks = img.to(device), masks.to(device)
                output = model(img)
                loss = loss_function(output, masks)
                val_loss += loss.item()
                prediction = torch.sigmoid(output) >= 0.5
                correct_pixels += (prediction == masks).sum().item()
                total_pixels += torch.numel(prediction)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = correct_pixels / total_pixels

        final_train_loss = avg_train_loss
        final_val_loss = avg_val_loss
        final_val_acc = avg_val_acc

        if (epoch + 1) % 1 == 0 or epoch == 0:
            print(f"Loss after iteration {epoch+1}: {avg_train_loss}")

    print("Final statistics")
    print(f"Final training loss: {final_train_loss}")
    print(f"Final validation loss: {final_val_loss}")
    print(f"Final validation accuracy: {final_val_acc*100} %")

In [ ]:
def setup_folders():
    folders = ['prediction_input', 'prediction_result']

    for f in folders:
        Path(f).mkdir(parents=True, exist_ok=True)

    print(f"Folder has been created: {folders}")

In [ ]:
def generate_weight_mask(tile_size):
    mask = np.ones((tile_size, tile_size), dtype=np.float32) #np.ones((h ,w), data type)
    mask = cv2.copyMakeBorder(mask[1:-1, 1:-1], 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0) #cv2.copyMakeBorder(target, up, down, left, right, mode, add data)
    mask = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5) #cv2.distanceTransform(target, mode, mask size) #target輸入須為uint8
    mask = mask / mask.max()

    return mask

In [ ]:
def prediction_model(input_path, model_path, output_path, tile_size=256, overlap=60):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if not os.path.exists(output_path):
        os.makedirs(output_path)

    file_name = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_h5 = os.path.join(output_path, f"{file_name}.h5")

    model = UNet(in_channels=1, out_channels=1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device)) #loading權重紀錄
    model.eval()

    image_extensions = ['.jpg', '.jpeg', '.tif', '.png']
    image_files = [f for f in os.listdir(input_path)
                   if Path(f).suffix.lower() in image_extensions]

    if not image_files:
        print("image not exist")
        return

    print(f"total image : {len(image_files)}")

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    stride = tile_size - overlap
    weight_mask = generate_weight_mask(tile_size)

    with h5py.File(output_h5, 'w') as h5f:
        for img_name in image_files:
            img_path = os.path.join(input_path, img_name)
            raw_img = cv2.imread(img_path) #解碼成np.array
            if raw_img is None: continue

            gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY) #為出原圖(給後處理heat map用), 此處才轉雙通道
            denoised = cv2.bilateralFilter(gray, 5, 75, 75)
            enhanced = clahe.apply(denoised)

            h, w = enhanced.shape #image size #(h, w) = (row, col) = (1040, 1388)

            #compute padding
            pad_h = (stride - (h - tile_size) % stride) % stride
            pad_w = (stride - (w - tile_size) % stride) % stride
            padded_img = cv2.copyMakeBorder(enhanced, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            #cv2.copyMakeBorder(image, 上邊增加數, 下邊增加數, 左邊增加數, 右邊增加數, 指定模式, 顏色)

            new_h, new_w = padded_img.shape
            prob_map = np.zeros((new_h, new_w), dtype=np.float32)
            count_map = np.zeros((new_h, new_w), dtype=np.float32) #儲存是否標記overlap

            print(f"lading: {img_name}")

            for y in range(0, new_h - tile_size + 1, stride):
                for x in range(0, new_w - tile_size + 1, stride):
                    tile = padded_img[y:y+tile_size, x:x+tile_size]

                    tile_t = torch.from_numpy(tile).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
                    #torch.from_numpy()自numpy轉pytorch tensor #(Batch, Channel, Height, Width) = (1, 1, 256, 256)

                    with torch.no_grad():
                        output = model(tile_t)
                        prob = torch.sigmoid(output).cpu().squeeze().numpy()

                    prob_map[y:y+tile_size, x:x+tile_size] += (prob * weight_mask)
                    count_map[y:y+tile_size, x:x+tile_size] += weight_mask

            final_prob = np.divide(prob_map, count_map, where=count_map != 0, out=np.zeros_like(prob_map))
            final_prob = final_prob[:h, :w] #[:h, :w]slice from 0 to h and w (delete padding)

            group = h5f.create_group(img_name)
            group.create_dataset('raw_image', data=raw_img)
            group.create_dataset('probability_map', data=final_prob)
            group.attrs['original_size'] = (h, w) #標記數據成分標籤

    print(f"{file_name} create complete")

In [ ]:
if __name__ == '__main__':
    mode = 'prediction'

    if mode == 'train':
        path = r'E:\cell_count\cell_dataset.h5'
        train_model(path, batch=16, epochs=60, seed=77, learning_rate=0.0001)
    if mode == 'prediction':
        setup_folders()
        prediction_model(input_path = 'prediction_input',
                         model_path = 'unet_cellcount_model.pth',
                         output_path = 'prediction_results')